# Transformers Text Classification Benchmark (Vietnamese)

Notebook này huấn luyện và so sánh nhiều mô hình Transformers cho bài toán text classification với dữ liệu đã được chia sẵn:
- `06_models/outputs/splits/train.jsonl`
- `06_models/outputs/splits/val.jsonl`
- `06_models/outputs/splits/test.jsonl`

## Mục tiêu
- Pipeline end-to-end: đọc train/val/test -> tokenize -> train/eval/test -> tổng hợp kết quả.
- Benchmark nhiều model trong một notebook.
- Tối ưu runtime theo hướng cân bằng tốc độ và theo dõi chi tiết.
- Tự động tận dụng GPU/multi-GPU nếu hệ thống hỗ trợ (qua Hugging Face Trainer).


In [1]:
# Nếu thiếu package, bỏ comment và chạy cell này
!pip install -qU wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 67.5 MB/s eta 0:00:00


In [ ]:
!pip install -qU huggingface_hub
!hf auth login --token <your_huggingface_token>

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 66.9 MB/s eta 0:00:00
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `public_token` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `public_token`


In [3]:
import json
import os
import random
import re
import time
import unicodedata
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

try:
    import wandb
except Exception:
    wandb = None


In [ ]:
@dataclass
class CFG:
    # Local split files (co san perplexity_score)
    train_path: str = "/kaggle/input/datasets/hongkimgip/check-worthiness-split/train.jsonl"
    val_path: str = "/kaggle/input/datasets/hongkimgip/check-worthiness-split/val.jsonl"
    test_path: str = "/kaggle/input/datasets/hongkimgip/check-worthiness-split/test.jsonl"
    output_root: str = "results"

    # Benchmark nhieu model
    model_names: tuple = (
        # "vinai/phobert-base",
        # "xlm-roberta-base",
        # "bert-base-multilingual-cased",
        "FPTAI/vibert-base-cased"
    )

    text_key: str = "text"
    label_key: str = "label"
    ppl_key: str = "perplexity_score"
    allowed_labels: tuple = (0, 1)

    seed: int = 42
    max_length: int = 256
    num_train_epochs: int = 3
    train_batch_size: int = 32
    eval_batch_size: int = 32
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    grad_accum_steps: int = 1

    # Che do toi uu thoi gian chay
    quick_run: bool = False
    quick_max_samples: int = 10000
    quick_max_train_steps: int = -1  # -1 = train du epoch

    # Logging/Eval
    logging_steps: int = 500
    eval_strategy: str = "epoch"
    save_strategy: str = "epoch"
    metric_for_best_model: str = "macro_f1"
    early_stopping_patience: int = 2

    # Binning metrics
    length_bin_count: int = 10
    ppl_bin_count: int = 10
    binning_strategy: str = "quantile"
    use_train_bins_for_all: bool = True
    log_train_bins: bool = True

    # Mixed precision
    use_fp16: bool = False
    use_bf16: bool = False

    # WandB
    use_wandb: bool = True
    wandb_project: str = "project-wiki-text-cls-benchmark-4-5"
    wandb_entity: str | None = None
    wandb_mode: str = "online"  # online/offline/disabled
    wandb_tags: tuple = ("notebook", "benchmark", "transformers")
    wandb_api_key: str = "<your_wandb_api_key>"

CFG = CFG()
set_seed(CFG.seed)

print(CFG)

CFG(train_path='/kaggle/input/datasets/hongkimgip/check-worthiness-split/train.jsonl', val_path='/kaggle/input/datasets/hongkimgip/check-worthiness-split/val.jsonl', test_path='/kaggle/input/datasets/hongkimgip/check-worthiness-split/test.jsonl', output_root='results', model_names=('google/electra-base-discriminator',), text_key='text', label_key='label', ppl_key='perplexity_score', allowed_labels=(0, 1), seed=42, max_length=256, num_train_epochs=3, train_batch_size=32, eval_batch_size=32, learning_rate=2e-05, weight_decay=0.01, warmup_ratio=0.1, grad_accum_steps=1, quick_run=False, quick_max_samples=10000, quick_max_train_steps=-1, logging_steps=500, eval_strategy='epoch', save_strategy='epoch', metric_for_best_model='macro_f1', early_stopping_patience=2, length_bin_count=10, ppl_bin_count=10, binning_strategy='quantile', use_train_bins_for_all=True, log_train_bins=True, use_fp16=False, use_bf16=False, use_wandb=True, wandb_project='project-wiki-text-cls-benchmark-4-5', wandb_entity=N

In [5]:
def detect_runtime(cfg: CFG):
    cuda_available = torch.cuda.is_available()
    gpu_count = torch.cuda.device_count() if cuda_available else 0

    # bf16 ưu tiên trên GPU mới; fp16 phổ biến hơn. Không bật đồng thời.
    fp16 = bool(cfg.use_fp16 and cuda_available and not cfg.use_bf16)
    bf16 = bool(cfg.use_bf16 and cuda_available and not cfg.use_fp16)

    runtime = {
        "cuda_available": cuda_available,
        "gpu_count": gpu_count,
        "device": "cuda" if cuda_available else "cpu",
        "fp16": fp16,
        "bf16": bf16,
    }

    print("=== Runtime Summary ===")
    print(json.dumps(runtime, indent=2))
    if cuda_available:
        for idx in range(gpu_count):
            print(f"GPU {idx}: {torch.cuda.get_device_name(idx)}")
        if gpu_count > 1:
            print("[info] Trainer sẽ tự động tận dụng multi-GPU nếu có nhiều GPU khả dụng.")
    else:
        print("[warning] CUDA không khả dụng. Training sẽ chạy trên CPU và chậm hơn đáng kể.")

    return runtime

RUNTIME = detect_runtime(CFG)


=== Runtime Summary ===
{
  "cuda_available": true,
  "gpu_count": 2,
  "device": "cuda",
  "fp16": false,
  "bf16": false
}
GPU 0: Tesla T4
GPU 1: Tesla T4
[info] Trainer sẽ tự động tận dụng multi-GPU nếu có nhiều GPU khả dụng.


In [6]:
_BASIC_PUNCTUATION = set(".,;:!?\"'()[]{}-_/\\")


def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text))
    text = text.replace("\u200b", " ").replace("\ufeff", " ")

    filtered_chars = []
    for ch in text:
        if ch.isspace():
            filtered_chars.append(" ")
            continue

        category = unicodedata.category(ch)
        if category[0] in {"L", "N"} or ch in _BASIC_PUNCTUATION:
            filtered_chars.append(ch)

    text = "".join(filtered_chars)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_jsonl_robust(
    path: str,
    text_key: str = "text",
    label_key: str = "label",
    ppl_key: str = "perplexity_score",
    allowed_labels=(0, 1),
    quick_run: bool = False,
    quick_max_samples: int = 100000,
 ):
    records = []
    stats = {
        "total_lines": 0,
        "valid": 0,
        "invalid_json": 0,
        "missing_fields": 0,
        "empty_text": 0,
        "invalid_label": 0,
        "missing_ppl": 0,
        "invalid_ppl": 0,
    }

    path_obj = Path(path)
    if not path_obj.exists():
        raise FileNotFoundError(f"Data file not found: {path}")

    with path_obj.open("r", encoding="utf-8") as f:
        for line in f:
            stats["total_lines"] += 1
            raw = line.strip()
            if not raw:
                continue
            try:
                obj = json.loads(raw)
            except json.JSONDecodeError:
                stats["invalid_json"] += 1
                continue

            if text_key not in obj or label_key not in obj:
                stats["missing_fields"] += 1
                continue

            text = clean_text(obj[text_key])
            if not text:
                stats["empty_text"] += 1
                continue

            label = obj[label_key]
            if label not in allowed_labels:
                stats["invalid_label"] += 1
                continue

            ppl_raw = obj.get(ppl_key, None)
            if ppl_raw is None:
                stats["missing_ppl"] += 1
                ppl_val = np.nan
            else:
                try:
                    ppl_val = float(ppl_raw)
                    if not np.isfinite(ppl_val):
                        raise ValueError("non-finite ppl")
                except Exception:
                    stats["invalid_ppl"] += 1
                    ppl_val = np.nan

            text_len_ws = len(text.split())
            records.append({
                "text": text,
                "label": int(label),
                ppl_key: ppl_val,
                "text_len_ws": text_len_ws,
            })
            stats["valid"] += 1

    if not records:
        raise ValueError("No valid records found after filtering.")

    # Quick mode: lay mau ngau nhien de debug nhanh vong lap huan luyen
    if quick_run and len(records) > quick_max_samples:
        random.seed(42)
        records = random.sample(records, quick_max_samples)

    df = pd.DataFrame.from_records(records)
    return df, stats


train_df, train_stats = load_jsonl_robust(
    path=CFG.train_path,
    text_key=CFG.text_key,
    label_key=CFG.label_key,
    ppl_key=CFG.ppl_key,
    allowed_labels=CFG.allowed_labels,
    quick_run=CFG.quick_run,
    quick_max_samples=CFG.quick_max_samples,
 )

val_df, val_stats = load_jsonl_robust(
    path=CFG.val_path,
    text_key=CFG.text_key,
    label_key=CFG.label_key,
    ppl_key=CFG.ppl_key,
    allowed_labels=CFG.allowed_labels,
    quick_run=CFG.quick_run,
    quick_max_samples=max(10000, CFG.quick_max_samples // 4),
 )

test_df, test_stats = load_jsonl_robust(
    path=CFG.test_path,
    text_key=CFG.text_key,
    label_key=CFG.label_key,
    ppl_key=CFG.ppl_key,
    allowed_labels=CFG.allowed_labels,
    quick_run=CFG.quick_run,
    quick_max_samples=max(10000, CFG.quick_max_samples // 4),
 )

df = pd.concat([train_df, val_df, test_df], ignore_index=True)
load_stats = {
    "train": train_stats,
    "val": val_stats,
    "test": test_stats,
    "total_rows_after_filtering": int(len(df)),
}

print("=== Load Stats (Pre-split Data) ===")
print(json.dumps(load_stats, indent=2))
print("\nRows after filtering:", len(df))
print("Split sizes:", len(train_df), len(val_df), len(test_df))
train_df.head(3)

=== Load Stats (Pre-split Data) ===
{
  "train": {
    "total_lines": 964974,
    "valid": 964972,
    "invalid_json": 0,
    "missing_fields": 0,
    "empty_text": 2,
    "invalid_label": 0,
    "missing_ppl": 0,
    "invalid_ppl": 0
  },
  "val": {
    "total_lines": 120621,
    "valid": 120619,
    "invalid_json": 0,
    "missing_fields": 0,
    "empty_text": 2,
    "invalid_label": 0,
    "missing_ppl": 0,
    "invalid_ppl": 0
  },
  "test": {
    "total_lines": 120622,
    "valid": 120622,
    "invalid_json": 0,
    "missing_fields": 0,
    "empty_text": 0,
    "invalid_label": 0,
    "missing_ppl": 0,
    "invalid_ppl": 0
  },
  "total_rows_after_filtering": 1206213
}

Rows after filtering: 1206213
Split sizes: 964972 120619 120622


,text,label,perplexity_score,text_len_ws
0,Lúc bấy giờ triều đình nhà Đường ngày một suy ...,0,8.226562,35
1,Cậu là người siêng năng chịu Mình Thánh Chúa t...,0,53.843750,39
2,Một đề tài tranh luận chính trị mới bất ngờ nổ...,1,20.078125,23


In [7]:
# EDA nhẹ để kiểm tra nhanh chất lượng dữ liệu và độ lệch lớp
label_counts = df["label"].value_counts().sort_index()
label_ratio = (label_counts / len(df) * 100).round(2)

sample_for_len = df.sample(min(20000, len(df)), random_state=CFG.seed)
char_len = sample_for_len["text"].str.len()

print("=== Label Distribution ===")
for k, v in label_counts.items():
    print(f"Label {k}: {v:,} ({label_ratio[k]}%)")

print("\n=== Text Length (char, sampled) ===")
print(char_len.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))


=== Label Distribution ===
Label 0: 862,169 (71.48%)
Label 1: 344,044 (28.52%)

=== Text Length (char, sampled) ===
count    20000.00000
mean       138.70105
std         56.38343
min         32.00000
50%        131.00000
90%        220.00000
95%        244.00000
99%        275.00000
max        420.00000
Name: text, dtype: float64


In [8]:
# Dữ liệu đã được chia sẵn từ script split_dataset_stratified.py
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

def show_label_ratio(name: str, split_df: pd.DataFrame):
    ratio = (split_df["label"].value_counts(normalize=True).sort_index() * 100).round(2)
    print(f"{name} label ratio (%):")
    print(ratio)
    print("-")

print("train/val/test:", len(train_df), len(val_df), len(test_df))
show_label_ratio("train", train_df)
show_label_ratio("val", val_df)
show_label_ratio("test", test_df)


train/val/test: 964972 120619 120622
train label ratio (%):
label
0    71.48
1    28.52
Name: proportion, dtype: float64
-
val label ratio (%):
label
0    71.47
1    28.53
Name: proportion, dtype: float64
-
test label ratio (%):
label
0    71.49
1    28.51
Name: proportion, dtype: float64
-


In [9]:
def build_split_meta(split_df: pd.DataFrame, ppl_key: str) -> dict:
    return {
        "length": split_df["text_len_ws"].to_numpy(),
        "ppl": split_df[ppl_key].to_numpy(),
    }

split_meta = {
    "train": build_split_meta(train_df, CFG.ppl_key),
    "validation": build_split_meta(val_df, CFG.ppl_key),
    "test": build_split_meta(test_df, CFG.ppl_key),
}

def compute_quantile_edges(values: np.ndarray, n_bins: int) -> np.ndarray:
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0 or n_bins < 1:
        return np.array([0.0, 1.0])
    edges = np.quantile(vals, np.linspace(0.0, 1.0, n_bins + 1))
    edges = np.unique(edges)
    if edges.size < 2:
        edges = np.array([edges[0] - 1.0, edges[0] + 1.0])
    return edges

if CFG.binning_strategy != "quantile":
    raise ValueError(f"Unsupported binning_strategy: {CFG.binning_strategy}")

def build_edges_for_split(values: dict) -> dict:
    return {
        "len": compute_quantile_edges(values["length"], CFG.length_bin_count),
        "ppl": compute_quantile_edges(values["ppl"], CFG.ppl_bin_count),
    }

if CFG.use_train_bins_for_all:
    base_edges = build_edges_for_split(split_meta["train"])
    BIN_EDGES = {
        "train": base_edges,
        "validation": base_edges,
        "test": base_edges,
    }
else:
    BIN_EDGES = {
        split: build_edges_for_split(values)
        for split, values in split_meta.items()
    }


In [10]:
label_list = sorted(df["label"].unique().tolist())
label2id = {int(v): int(v) for v in label_list}
id2label = {int(v): str(v) for v in label_list}

raw_ds = DatasetDict({
    "train": Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False),
    "validation": Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False),
    "test": Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False),
})

raw_ds


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 964972
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 120619
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 120622
    })
})

In [11]:
import math
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    precision_recall_fscore_support,
 )

IMPORTANT_METRIC_KEYS = {
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "precision",
    "recall",
    "f1",
    "precision_neg",
    "recall_neg",
    "f1_neg",
    "support_neg",
    "precision_pos",
    "recall_pos",
    "f1_pos",
    "support_pos",
}


def logits_to_preds(logits: np.ndarray) -> np.ndarray:
    if logits.ndim == 1 or logits.shape[-1] == 1:
        probs = 1 / (1 + np.exp(-logits.reshape(-1)))
        return (probs > 0.5).astype(int)
    return np.argmax(logits, axis=-1)


def compute_basic_metrics(labels: np.ndarray, preds: np.ndarray) -> dict:
    # Overall metrics
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(labels, preds, average="weighted", zero_division=0)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    # Per-class metrics (explicit order: [0, 1])
    precision_cls, recall_cls, f1_cls, support_cls = precision_recall_fscore_support(
        labels,
        preds,
        labels=[0, 1],
        average=None,
        zero_division=0,
    )

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "precision_neg": precision_cls[0],
        "recall_neg": recall_cls[0],
        "f1_neg": f1_cls[0],
        "support_neg": support_cls[0],
        "precision_pos": precision_cls[1],
        "recall_pos": recall_cls[1],
        "f1_pos": f1_cls[1],
        "support_pos": support_cls[1],
    }


def compute_bin_metrics(
    labels: np.ndarray,
    preds: np.ndarray,
    values: np.ndarray,
    edges: np.ndarray,
    prefix: str,
) -> dict:
    metrics = {}
    values = np.asarray(values, dtype=float)
    finite_mask = np.isfinite(values)
    if edges is None or len(edges) < 2:
        return metrics

    bin_ids = np.digitize(values, edges[1:-1], right=False)
    n_bins = len(edges) - 1
    for b in range(n_bins):
        idx = (bin_ids == b) & finite_mask
        bin_key = f"{prefix}_bin_{b}"
        metrics[f"{bin_key}/edge_low"] = float(edges[b])
        metrics[f"{bin_key}/edge_high"] = float(edges[b + 1])
        metrics[f"{bin_key}/support_total"] = int(idx.sum())
        if not idx.any():
            for k in IMPORTANT_METRIC_KEYS:
                metrics[f"{bin_key}/{k}"] = float("nan")
            continue

        sub = compute_basic_metrics(labels[idx], preds[idx])
        for k, v in sub.items():
            metrics[f"{bin_key}/{k}"] = v
    return metrics


def prefix_metrics(metrics: dict, prefix: str) -> dict:
    return {f"{prefix}/{k}": v for k, v in metrics.items()}


def extract_important_split_metrics(eval_metrics: dict, split_prefix: str) -> dict:
    filtered = {}
    prefix = f"{split_prefix}_"
    for key in IMPORTANT_METRIC_KEYS:
        raw_key = f"{prefix}{key}"
        if raw_key in eval_metrics:
            filtered[key] = eval_metrics[raw_key]
    return filtered


def summarize_split_metrics(metrics: dict) -> dict:
    summary_keys = ["accuracy", "recall", "f1", "macro_f1", "weighted_f1"]
    return {k: metrics.get(k) for k in summary_keys}


def to_json_safe(value):
    if isinstance(value, np.ndarray):
        return [to_json_safe(v) for v in value.tolist()]
    if isinstance(value, np.generic):
        return to_json_safe(value.item())
    if isinstance(value, dict):
        return {str(k): to_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_json_safe(v) for v in value]
    if isinstance(value, float):
        if not math.isfinite(value):
            return None
        return value
    if isinstance(value, Path):
        return str(value)
    return value


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits_to_preds(logits)
    return compute_basic_metrics(labels, preds)


def sanitize_model_name(name: str) -> str:
    return name.replace("/", "_").replace(" ", "_").lower()


def build_training_args(cfg: CFG, run_dir: Path, run_name: str, runtime: dict):
    report_to = ["wandb"] if cfg.use_wandb and wandb is not None else []

    return TrainingArguments(
        output_dir=str(run_dir),
        num_train_epochs=cfg.num_train_epochs,
        learning_rate=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
        warmup_ratio=cfg.warmup_ratio,
        per_device_train_batch_size=cfg.train_batch_size,
        per_device_eval_batch_size=cfg.eval_batch_size,
        gradient_accumulation_steps=cfg.grad_accum_steps,
        eval_strategy=cfg.eval_strategy,
        save_strategy=cfg.save_strategy,
        logging_strategy="steps",
        logging_steps=cfg.logging_steps,
        load_best_model_at_end=True,
        metric_for_best_model=cfg.metric_for_best_model,
        greater_is_better=True,
        save_total_limit=1,
        fp16=runtime["fp16"],
        bf16=runtime["bf16"],
        dataloader_num_workers=0,
        dataloader_pin_memory=runtime["cuda_available"],
        report_to=report_to,
        run_name=run_name,
        max_steps=cfg.quick_max_train_steps if cfg.quick_run and cfg.quick_max_train_steps > 0 else -1,
    )

In [ ]:
def maybe_init_wandb(cfg: CFG):
    if not cfg.use_wandb:
        return
    if wandb is None:
        print("[warning] use_wandb=True nhung chua import duoc wandb; bo qua log W&B.")
        return

    os.environ.setdefault("WANDB_WATCH", "false")
    os.environ.setdefault("WANDB_LOG_MODEL", "false")

    # Login neu can (neu ban da login tu truoc thi lenh nay khong gay loi)
    try:
        wandb.login(key=CFG.wandb_api_key)
    except Exception as e:
        print(f"[warning] wandb.login() failed: {e}")

    # project name
    os.environ['WANDB_PROJECT'] = cfg.wandb_project


def build_tokenized_datasets(model_name: str, ds: DatasetDict, cfg: CFG):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    def tokenize_batch(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=cfg.max_length,
        )

    tokenized = ds.map(
        tokenize_batch,
        batched=True,
        remove_columns=["text"],
        desc=f"Tokenizing for {model_name}",
    )

    # Preflight validation mot mau batch de fail-fast
    sample = tokenized["train"][0]
    if "input_ids" not in sample or "label" not in sample:
        raise ValueError("Tokenized dataset missing required keys: input_ids/label")

    collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8 if RUNTIME["cuda_available"] else None)
    return tokenizer, tokenized, collator


def _predict_split(trainer: Trainer, split_ds: Dataset) -> tuple[np.ndarray, np.ndarray]:
    pred = trainer.predict(split_ds)
    logits = pred.predictions
    labels = pred.label_ids
    preds = logits_to_preds(logits)
    return labels, preds


def _compute_and_log_bin_metrics(
    trainer: Trainer,
    tokenized: DatasetDict,
    cfg: CFG,
    split_meta: dict,
    bin_edges: dict,
) -> dict:
    split_prefix = {"train": "train", "validation": "val", "test": "test"}
    bin_metrics_by_split = {"train": {}, "validation": {}, "test": {}}

    for split_name in ["train", "validation", "test"]:
        if split_name == "train" and not cfg.log_train_bins:
            continue

        labels, preds = _predict_split(trainer, tokenized[split_name])
        split_values = split_meta[split_name]
        edges = bin_edges[split_name]

        len_metrics = compute_bin_metrics(
            labels,
            preds,
            split_values["length"],
            edges["len"],
            "len",
        )
        ppl_metrics = compute_bin_metrics(
            labels,
            preds,
            split_values["ppl"],
            edges["ppl"],
            "ppl",
        )

        merged = {**len_metrics, **ppl_metrics}
        bin_metrics_by_split[split_name] = merged

        if cfg.use_wandb and wandb is not None and wandb.run is not None:
            wandb.log(prefix_metrics(merged, split_prefix[split_name]))

    return bin_metrics_by_split


def train_one_model(model_name: str, cfg: CFG, runtime: dict, ds: DatasetDict):
    set_seed(cfg.seed)

    safe_name = sanitize_model_name(model_name)
    run_ts = time.strftime("%Y%m%d_%H%M%S")
    run_name = f"{safe_name}_{run_ts}"
    run_dir = Path(cfg.output_root) / safe_name / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    tokenizer, tokenized, collator = build_tokenized_datasets(model_name, ds, cfg)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(label2id),
        label2id={str(k): int(v) for k, v in label2id.items()},
        id2label={int(k): str(v) for k, v in id2label.items()},
    )

    args = build_training_args(cfg, run_dir, run_name, runtime)

    callbacks = []
    if cfg.early_stopping_patience is not None and cfg.early_stopping_patience > 0:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=cfg.early_stopping_patience))

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=callbacks,
    )

    start = time.time()
    trainer.train()
    elapsed = time.time() - start

    val_eval_metrics = trainer.evaluate(tokenized["validation"], metric_key_prefix="val")
    test_eval_metrics = trainer.evaluate(tokenized["test"], metric_key_prefix="test")

    bin_metrics = _compute_and_log_bin_metrics(
        trainer=trainer,
        tokenized=tokenized,
        cfg=cfg,
        split_meta=split_meta,
        bin_edges=BIN_EDGES,
    )

    val_metrics = extract_important_split_metrics(val_eval_metrics, split_prefix="val")
    test_metrics = extract_important_split_metrics(test_eval_metrics, split_prefix="test")

    best_dir = run_dir / "best_model"
    trainer.save_model(str(best_dir))
    tokenizer.save_pretrained(str(best_dir))

    metrics = {
        "model_name": model_name,
        "run_name": run_name,
        "val_accuracy": val_metrics.get("accuracy"),
        "val_recall": val_metrics.get("recall"),
        "val_f1": val_metrics.get("f1"),
        "val_macro_f1": val_metrics.get("macro_f1"),
        "test_accuracy": test_metrics.get("accuracy"),
        "test_recall": test_metrics.get("recall"),
        "test_f1": test_metrics.get("f1"),
        "test_macro_f1": test_metrics.get("macro_f1"),
        "output_dir": str(run_dir),
    }

    metrics_payload = {
        "meta": {
            "model_name": model_name,
            "run_name": run_name,
            "output_dir": str(run_dir),
            "train_seconds": round(elapsed, 2),
        },
        "summary": {
            "validation": summarize_split_metrics(val_metrics),
            "test": summarize_split_metrics(test_metrics),
        },
        "metrics_by_split": {
            "validation": val_metrics,
            "test": test_metrics,
        },
        "bin_metrics": bin_metrics,
        "bin_edges": BIN_EDGES,
    }

    with (run_dir / "metrics.json").open("w", encoding="utf-8") as f:
        json.dump(
            to_json_safe(metrics_payload),
            f,
            ensure_ascii=False,
            indent=2,
            allow_nan=False,
        )

    # end wandb session
    if wandb is not None and wandb.run is not None:
        wandb.finish()

    return metrics, trainer, tokenizer


maybe_init_wandb(CFG)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hoangkimgiap_t67 (hoangkimgiap_t67-vietnam-national-university-hanoi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [13]:
all_results = []
trained_objects = {}

for model_name in CFG.model_names:
    print(f"\n===== Training: {model_name} =====")
    try:
        metrics, trainer, tokenizer = train_one_model(
            model_name=model_name,
            cfg=CFG,
            runtime=RUNTIME,
            ds=raw_ds,
        )
        all_results.append(metrics)
        trained_objects[model_name] = {"trainer": trainer, "tokenizer": tokenizer}
        print(f"[done] {model_name}: val_macro_f1={metrics['val_macro_f1']:.4f}, test_macro_f1={metrics['test_macro_f1']:.4f}")
    except Exception as e:
        print(f"[error] {model_name}: {e}")

results_df = pd.DataFrame(all_results)
if not results_df.empty:
    results_df = results_df.sort_values(by=["test_macro_f1", "val_macro_f1"], ascending=False).reset_index(drop=True)

results_df



===== Training: google/electra-base-discriminator =====


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml; uploading output.log
wandb: uploading history steps 100-101, summary
wandb: 
wandb: Run history:
wandb:      eval/accuracy ▁▆█
wandb:            eval/f1 ▁█▇
wandb:        eval/f1_neg ▁▅█
wandb:        eval/f1_pos ▁█▇
wandb:          eval/loss █▁▁
wandb:      eval/macro_f1 ▁██
wandb:     eval/precision ▁▅█
wandb: eval/precision_neg ▁█▇
wandb: eval/precision_pos ▁▅█
wandb:        eval/recall ▁█▆
wandb:             +1,069 ...
wandb: 
wandb: Run summary:
wandb:      eval/accuracy 0.75537
wandb:            eval/f1 0.45654
wandb:        eval/f1_neg 0.84216
wandb:        eval/f1_pos 0.45654
wandb:          eval/loss 1.04068
wandb:      eval/macro_f1 0.64935
wandb:     eval/precision 0.62328
wandb: eval/precision_neg 0.78144
wandb: eval/precision_pos 0.62328
wandb:        eval/recall 0.36019
wandb:             +1,074 ...
wandb: 
wandb: 🚀 View run google_electra-base-discriminator_20260506_034953 at: https:

[done] google/electra-base-discriminator: val_macro_f1=0.6511, test_macro_f1=0.6508


,model_name,run_name,val_accuracy,val_recall,val_f1,val_macro_f1,test_accuracy,test_recall,test_f1,test_macro_f1,output_dir
0,google/electra-base-discriminator,google_electra-base-discriminator_20260506_034953,0.753256,0.371723,0.462236,0.651067,0.753561,0.370297,0.461466,0.650844,results/google_electra-base-discriminator/goog...


In [14]:
# Lưu bảng so sánh tổng hợp
Path(CFG.output_root).mkdir(parents=True, exist_ok=True)
summary_csv = Path(CFG.output_root) / "benchmark_summary.csv"
summary_json = Path(CFG.output_root) / "benchmark_summary.json"

if not results_df.empty:
    results_df.to_csv(summary_csv, index=False)
    with summary_json.open("w", encoding="utf-8") as f:
        json.dump(results_df.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

print(f"Saved summary CSV: {summary_csv}")
print(f"Saved summary JSON: {summary_json}")


Saved summary CSV: results/benchmark_summary.csv
Saved summary JSON: results/benchmark_summary.json


In [15]:
# Inference nhanh với model tốt nhất
if results_df.empty:
    print("Chưa có model train thành công để inference.")
else:
    best_model_name = results_df.iloc[0]["model_name"]
    best_trainer = trained_objects[best_model_name]["trainer"]
    best_tokenizer = trained_objects[best_model_name]["tokenizer"]

    sample_texts = [
        "Nghiên cứu mới cho thấy chất lượng không khí tại khu vực nội đô đã cải thiện trong năm nay.",
        "Theo [1], phương pháp này đạt hiệu quả cao hơn khi dữ liệu được cân bằng đúng cách.",
    ]
    sample_texts = [clean_text(t) for t in sample_texts]

    enc = best_tokenizer(sample_texts, truncation=True, max_length=CFG.max_length, padding=True, return_tensors="pt")
    device = best_trainer.model.device
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = best_trainer.model(**enc).logits
        preds = torch.argmax(logits, dim=-1).detach().cpu().tolist()

    for t, p in zip(sample_texts, preds):
        print(f"pred={p} | text={t}")

pred=0 | text=Nghiên cứu mới cho thấy chất lượng không khí tại khu vực nội đô đã cải thiện trong năm nay.
pred=1 | text=Theo [1], phương pháp này đạt hiệu quả cao hơn khi dữ liệu được cân bằng đúng cách.
